In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import funcoes
import unicodedata

pd.options.display.float_format = '{:.2f}'.format #para os números aparecerem com duas casas decimais
pd.set_option('display.max_columns', 50) #mostra 50 colunas ou o número desejado
pd.set_option('future.no_silent_downcasting', True)

dados1 = pd.read_excel("denuncias.xlsx")
dados2 = pd.read_excel("denuncias-2024.xlsx")
dados3 = pd.read_excel("denuncias-2025.xlsx")



seg = dados1.copy()
seg2 = dados2.copy()
seg3 = dados3.copy()



In [2]:
lista_dados = [seg, seg2, seg3]
for dados in lista_dados:
    funcoes.padronizar_colunas(dados)

In [3]:
seg3 = seg3.drop(columns=['obs.'])

In [4]:
#renomeando colunas no df3
seg3 = seg3.rename(columns={'nº_protocolo': 'protocolo'})


In [5]:
#nova ordem das colunas seg3 
nova_ordem = ['protocolo', 'data', 'nome_da_vítima', 'tipo_de_violência_denunciada', 'bairro_da_vítima', 'faixa_etária', 'raça/cor', 'sexo', 'vínculo_do_suspeito', 'devolutiva']
seg3 = seg3[nova_ordem]

In [6]:
#passando a mesma lista de colunas para os três dfs
lista_dados = [seg, seg2, seg3]
nova_ordem = ['protocolo', 'data', 'nome_da_vítima', 'tipo_de_violência_denunciada', 'bairro_da_vítima', 'faixa_etária', 'raça/cor', 'sexo', 'vínculo_do_suspeito', 'devolutiva']
for df in lista_dados:
    df.columns = nova_ordem

In [7]:
lista_dados = [seg, seg2, seg3]
for dados in lista_dados:
    funcoes.resumo(dados)

--- Formato ---
(94, 10)

--- Tipos de Dados ---
protocolo                                int64
data                            datetime64[ns]
nome_da_vítima                          object
tipo_de_violência_denunciada            object
bairro_da_vítima                        object
faixa_etária                            object
raça/cor                                object
sexo                                    object
vínculo_do_suspeito                     object
devolutiva                              object
dtype: object

--- Valores Nulos ---
protocolo                        0
data                             0
nome_da_vítima                   0
tipo_de_violência_denunciada     0
bairro_da_vítima                 0
faixa_etária                     0
raça/cor                         0
sexo                             0
vínculo_do_suspeito              0
devolutiva                      61
dtype: int64

--- Resumo Estatístico ---
--- Formato ---
(146, 10)

--- Tipos de Dados ---
pro

In [8]:
df_completo = pd.concat([seg, seg2, seg3], ignore_index=False)
df_completo

,protocolo,data,nome_da_vítima,tipo_de_violência_denunciada,bairro_da_vítima,faixa_etária,raça/cor,sexo,vínculo_do_suspeito,devolutiva
0,1681957.00,2023-03-14 00:00:00,Nelson Lucílio,Patrimonial,Jardim Bertioga,65 a 69 anos,Parda,Masculino,Filhas,Não consta nos registros da GPSEMC a resposta
1,1690353.00,2023-03-18 00:00:00,Maura Rosa Rosin,Negligência / Patrimonial,Vila Santa Izabel,75 a 79 anos,Branca,Feminino,Filho,25/07/23 - CREAS realizou visita domiciliar po...
2,1690575.00,2023-03-18 00:00:00,José Aparecido de Oliveira Santos,Psicológica; Patrimonial,Parque Tarumã,65 a 69 anos,Parda,Masculino,Estado do Paraná,17/04/2023 -pessoa idosa encontra-se em acompa...
3,1690834.00,2023-03-18 00:00:00,Maria Amélia Soares Bovo,Psicológica; Patrimonial,Zona 05,80 a 84 anos,Branca,Feminino,Neta,16/06/2023 - CREAS realizou visita domiciliar ...
4,1697158.00,2023-03-21 00:00:00,Weslei Pereira dos Santos,Patrimonial; Maus Tratos,Zona 04,75 a 79 anos,Branca,Masculino,Companheira,19/05/2023 - inseriu em acompanhamento PAEFI
...,...,...,...,...,...,...,...,...,...,...
91,4152733.00,2025-12-12 00:00:00,Josina,"Física, Psicológica/Psíquica, Negligência/Aban...",Jardim da Glória,80+,Preta,Fem,Filho(a),NaN
92,4167778.00,2025-12-17 00:00:00,Izildinha Maria Sanxes,"Psicológica/Psíquica, Negligência/Abandono",Jardim Dourados,60+,Branca,Fem,Filho(a),NaN
93,4174995.00,2025-12-19 00:00:00,Marilena Busnelo,"Física, Psicológica/Psíquica, Negligência/Aban...",Vila Esperança,70+,Branca,Fem,Outros,NaN
94,4185244.00,2025-12-26 00:00:00,Luzia Celestina de Jesus,"Psicológica/Psíquica, Patrimonial, Negligência...",NaN,80+,Preta,Fem,NaN,NaN


In [9]:
#faltou tratar as colunas de vínculo e bairro, a de vínculo precisa de mais tratamento. 
df_completo['sexo'].unique()

df_completo['sexo'] = df_completo['sexo'].replace({
    'Fem': 'Feminino', 
    'Masc': 'Masculino',
    'Não Informado': 'Não informado'
})

In [10]:
df_completo['faixa_etária'].unique()

df_completo['faixa_etária'] = df_completo['faixa_etária'].replace({
    '65 a 69 anos': '65 a 69', 
    '75 a 79 anos': '75 a 79',
    '80 a 84 anos': '80 a 84',
    '90+': '90 a 94', 
    '60 a 64 anos': '60 a 64',
    '85 a 89 anos': '85 a 89',
    '70 a 74 anos': '70 a 74',
    '75 a 79 anos ': '75 a 79',
    '70 a 74 anos ': '70 a 74',
    '60 à 70': '60 a 64',
    '60 a 64 anos ': '60 a 64',
    '80+': '80 a 84',
    '70+': '70 a 74',
    '60+': '60 a 64'
})

In [11]:
df_completo['vínculo_do_suspeito'].unique()

df_completo['vínculo_do_suspeito'] = df_completo['vínculo_do_suspeito'].replace({
    'Filhas': 'Filha(o)',
    'Filho e Nora': 'Filha(o)',
    'Filhos': 'Filha(o)',
    'Companheiro e sua Amante': 'Companheiro(a)',
    'Companheira': 'Companheiro(a)',
    'Companheiro ': 'Companheiro(a)',
    'Companheiro e Filho': 'Companheiro(a)',
    'Filha': 'Filha(o)',
    'Prefeitura de Maringá': 'Instituição Pública',
    'Ministério Público': 'Instituição Pública',
    'Viação TCC': 'Instituição Privada', 
    'CRAS': 'Instituição Pública',
    'Segurança Comercial': 'Instituição Privada',
    'Dono do Pensionato e os Filhos ': 'Dono Pensionato',
    'Família com quem convive': 'Familiar',
    'Condomínio/Síndico': 'Síndico',
    'Companheiro': 'Companheiro(a)',
    '9° SDP': 'Instituição Pública',
    'Filhos, Companheira e Neta': 'Familiar',
    'Filho de criação': 'Filha(o)',
    'Copel': 'Instituição Pública',
    'Sobrinha do Companheiro': 'Familiar', 
    'Companheiro, Filho e Nora': 'Familiar',
    'Sobrinha e Filhos': 'Filha(o)',
    'Irmã': 'Irmã(o)',
    'Filha e Netas': 'Filha(o)',
    'Instituição': 'Instituição Pública',
    'Secretaria de Saúde': 'Instituição Pública',
    'Tia da Ex-Nora': 'Familiar',
    'Empresa': 'Instituição Privada',
    'Primo': 'Familiar',
    'Irmã e Sobrinha': 'Irmã(o)',
    'FIlho': 'Filha(o)', 
    'Familiares': 'Familiar', 
    'Filho(a)': 'Filha(o)',
    'Sobrinho': 'Familiar', 
    'Irmãos': 'Irmã(o)',
    'Profissional de Saúde': 'Instituição Pública',
    'Estabelecimento de Saúde': 'Instituição Pública',
    'Companheira e Filha': 'Filha(o)',
    'FIlha': 'Filha(o)',
    'Vendedor de Passagem': 'Instituição Privada',
    'FIlhos': 'Filha(o)',
    'Sobrinha': 'Familiar',
    'Hospital': 'Instituição Privada',
    'Não Informado': 'Não informado',
    'Hospital Bom Samaritano': 'Instituição Privada',
    'Neta e Filha': 'Filha(o)',
    'Clínica Doutor Atende': 'Instituição Privada',
    'Polícia Militar': 'Instituição Pública',
    'Filho e Cuidadora': 'Filha(o)', 
    'Filho e Ex-Nora': 'Filha(o)', 
    'TCCC': 'Instituição Privada',
    'Hospital Santa Rita': 'Instituição Privada',
    'Imobiliária Me Construtora Casa Verde' : 'Instituição Privada',
    'Sobrinhos': 'Familiar',
    'Filhos ': 'Filha(o)',
    'Plano Humana Saúde': 'Instituição Privada',
    'Clínica Vida Ativa': 'Instituição Privada',
    'Prostitutas': 'Instituição Privada',
    'Empregador/Patrão': 'Empregador', 
    'Asilo São Vicente de Paula': 'Instituição Privada',
    'Irmãs': 'Irmã(o)',
    'Filho e Companheira': 'Filha(o)',
    'Irmão e Cunhada': 'Irmã(o)',
    'UPA Zona Norte': 'Instituição Pública',
    'Supermercado Muffato': 'Instituição Privada',
    'Hotel Elo': 'Instituição Privada',
    'Diretora/Gestora de Instituição / Casa Lar Benedito Franchini': 'Instituição Privada',
    'Genro': 'Familiar',
    'Genro e Filha': 'Familiar',
    'COPEL': 'Instituição Privada', 
    'Esposa': 'Companheiro(a)', 
    'Filho(a) ': 'Filha(o)',
    'Pessoa com quem mantém/manteve convivência familiar;': 'Familiar'
    
    
})

In [12]:
df_completo['vínculo_do_suspeito'] = df_completo['vínculo_do_suspeito'].replace({
    'Filho': 'Filha(o)',
    'Estado do Paraná': 'Instituição Pública',
    'Neta': 'Neto(a)',
    'Neto': 'Neto(a)',
    'Ex-Nora': 'Familiar',
    'Nora': 'Familiar',
    'Irmão': 'Irmã(o)',
    'Vizinha': 'Vizinhos',
    
    
})


In [13]:
df_completo['vínculo_do_suspeito'].unique()

array(['Filha(o)', 'Instituição Pública', 'Neto(a)', 'Companheiro(a)',
       'Não informado', 'Não Possui', 'Instituição Privada', 'Familiar',
       'Amante do Companheiro', 'Vizinho', 'Dono Pensionato', 'Síndico',
       'Irmã(o)', 'Ex-Companheira', 'Amigo', 'Vizinhos', 'Ex-Companheiro',
       'Instituição clandestina', 'Cuidadora', 'Empregador',
       'Não se aplica', 'Inquilinas', 'Não se aplica;', 'Outros', nan,
       'Desconhecido(a)'], dtype=object)

In [14]:
df_completo['data'] = pd.to_datetime(df_completo['data'], errors='coerce')
df_completo['ano'] = df_completo['data'].dt.year
df_completo


,protocolo,data,nome_da_vítima,tipo_de_violência_denunciada,bairro_da_vítima,faixa_etária,raça/cor,sexo,vínculo_do_suspeito,devolutiva,ano
0,1681957.00,2023-03-14,Nelson Lucílio,Patrimonial,Jardim Bertioga,65 a 69,Parda,Masculino,Filha(o),Não consta nos registros da GPSEMC a resposta,2023.00
1,1690353.00,2023-03-18,Maura Rosa Rosin,Negligência / Patrimonial,Vila Santa Izabel,75 a 79,Branca,Feminino,Filha(o),25/07/23 - CREAS realizou visita domiciliar po...,2023.00
2,1690575.00,2023-03-18,José Aparecido de Oliveira Santos,Psicológica; Patrimonial,Parque Tarumã,65 a 69,Parda,Masculino,Instituição Pública,17/04/2023 -pessoa idosa encontra-se em acompa...,2023.00
3,1690834.00,2023-03-18,Maria Amélia Soares Bovo,Psicológica; Patrimonial,Zona 05,80 a 84,Branca,Feminino,Neto(a),16/06/2023 - CREAS realizou visita domiciliar ...,2023.00
4,1697158.00,2023-03-21,Weslei Pereira dos Santos,Patrimonial; Maus Tratos,Zona 04,75 a 79,Branca,Masculino,Companheiro(a),19/05/2023 - inseriu em acompanhamento PAEFI,2023.00
...,...,...,...,...,...,...,...,...,...,...,...
91,4152733.00,2025-12-12,Josina,"Física, Psicológica/Psíquica, Negligência/Aban...",Jardim da Glória,80 a 84,Preta,Feminino,Filha(o),NaN,2025.00
92,4167778.00,2025-12-17,Izildinha Maria Sanxes,"Psicológica/Psíquica, Negligência/Abandono",Jardim Dourados,60 a 64,Branca,Feminino,Filha(o),NaN,2025.00
93,4174995.00,2025-12-19,Marilena Busnelo,"Física, Psicológica/Psíquica, Negligência/Aban...",Vila Esperança,70 a 74,Branca,Feminino,Outros,NaN,2025.00
94,4185244.00,2025-12-26,Luzia Celestina de Jesus,"Psicológica/Psíquica, Patrimonial, Negligência...",NaN,80 a 84,Preta,Feminino,NaN,NaN,2025.00


In [15]:
#retirando caracteres como /, e, ;, -, 
df_completo['tipo_de_violência_denunciada'].unique()
df_completo['tipo_de_violência_denunciada'] = df_completo['tipo_de_violência_denunciada'].str.replace(r'[/;]', ', ', regex=True)
#alterando " e " por ', ' 
df_completo['tipo_de_violência_denunciada'] = df_completo['tipo_de_violência_denunciada'].str.replace(r'\s+e\s+', ', ', regex=True)

df_completo['tipo_de_violência_denunciada'] = df_completo['tipo_de_violência_denunciada'].str.replace(r'-', ' ', regex=True)

In [16]:
df_completo = df_completo.drop(['nome_da_vítima', 'devolutiva'], axis=1)


In [17]:
#df_exp, expandindo os tipos de violencia denunciada
df_completo = df_completo.assign(tipo_de_violência_denunciada=df_completo['tipo_de_violência_denunciada'].str.split(',')).explode('tipo_de_violência_denunciada')


In [29]:
df_completo = df_completo.dropna(subset=['ano'])
df_completo = df_completo.dropna(subset=['bairro_da_vítima'])

In [30]:
df_completo['protocolo'] = df_completo['protocolo'].astype(str)
df_completo['ano'] = df_completo['ano'].astype(int)
df_completo['tipo_de_violência_denunciada'] = df_completo['tipo_de_violência_denunciada'].str.strip()
df_completo

,protocolo,data,tipo_de_violência_denunciada,bairro_da_vítima,faixa_etária,raça/cor,sexo,vínculo_do_suspeito,ano
0,1681957.0,2023-03-14,Patrimonial,Jardim Bertioga,65 a 69,Parda,Masculino,Filha(o),2023
1,1690353.0,2023-03-18,Negligência,Vila Santa Izabel,75 a 79,Branca,Feminino,Filha(o),2023
1,1690353.0,2023-03-18,Patrimonial,Vila Santa Izabel,75 a 79,Branca,Feminino,Filha(o),2023
2,1690575.0,2023-03-18,Psicológica,Parque Tarumã,65 a 69,Parda,Masculino,Instituição Pública,2023
2,1690575.0,2023-03-18,Patrimonial,Parque Tarumã,65 a 69,Parda,Masculino,Instituição Pública,2023
...,...,...,...,...,...,...,...,...,...
93,4174995.0,2025-12-19,Psíquica,Vila Esperança,70 a 74,Branca,Feminino,Outros,2025
93,4174995.0,2025-12-19,Negligência,Vila Esperança,70 a 74,Branca,Feminino,Outros,2025
93,4174995.0,2025-12-19,Abandono,Vila Esperança,70 a 74,Branca,Feminino,Outros,2025
95,4190066.0,2025-12-30,Psicológica,Zona 7,70 a 74,Branca,Feminino,Desconhecido(a),2025


In [33]:
df_completo['tipo_de_violência_denunciada'].unique()
df_completo['tipo_de_violência_denunciada'] = df_completo['tipo_de_violência_denunciada'].replace({
    'Negligência ': 'Negligência',
    '  Patrimonial': 'Patrimonial',
    '  Maus Tratos': 'Maus Tratos',
    '  Cárcere Privado': 'Cárcere Privado',
    '  Psicológica': 'Psicológica',
    '  Verbal': 'Verbal',
    '  Psicológica ': 'Psicológica',
    '  Verbal': 'Verbal', 
    '  Psicológica ': 'Psicológica', 
    '  Patrimonial ': 'Patrimonial', 
    '  Negligência': 'Negligência',
    ' Verbal': 'Verbal',
    '  Física': 'Física', 
    ' Negligência': 'Negligência', 
    ' Psicológica': 'Psicológica', 
    ' Cárcere Privado': 'Cárcere Privado',
    ' Física': 'Física',
    ' Negligência': 'Negligência', 
    ' Psicológica': 'Psicológica', 
    ' Cárcere Privado': 'Cárcere Privado',
    ' Maus Tratos': 'Maus Tratos', 
    ' Patrimonial': 'Patrimonial', 
    ' Psicológica ': 'Psicológica', 
    ' Direitos Violados': 'Direitos Violados', 
    'Neglicência': 'Negligencia', 
    ' Patriomonial': 'Patrimonial',
    ' Sexual': 'Sexual', 
    'Física ': 'Física', 
    ' Maus tratos': 'Maus Tratos', 
    'Negligência Maus Tratos': 'Negligência', 
    'Psicológica ': 'Psicológica', 
    ' Patrimonial ': 'Patrimonial', 
    'Direitos': 'Direitos Violados',
    ' Maus Tratos ': 'Maus Tratos', 
    ' Psicológico': 'Psicológica',
    'Piscológica': 'Psicológica',
    ' Discriminação': 'Discriminação',
    ' Psíquica': 'Psicológica', 
    ' Abandono': 'Abandono', 
    ' Moral': 'Moral', 
    ' Intolerância': 'Intolerância', 
    ' Direitos Civis': 'Direitos Civis', 
    'Políticos': 'Direitos Políticos',
    'Negligencia': 'Negligência',
})

In [34]:
df_completo['tipo_de_violência_denunciada'].unique()

array(['Patrimonial', 'Negligência', 'Psicológica', 'Maus Tratos',
       'Verbal', 'Cárcere Privado', 'Física', 'Direitos Violados',
       'Patriomonial', 'Sexual', 'Maus tratos', 'Psicológico',
       'Discriminação', 'Psíquica', 'Abandono', 'Moral', 'Intolerância',
       'Direitos Civis', 'Direitos Políticos'], dtype=object)

In [37]:
df_completo.to_excel('denuncias-2023-2025.xlsx', index=False)

In [38]:
funcoes.resumo(df_completo)

--- Formato ---
(1007, 9)

--- Tipos de Dados ---
protocolo                               object
data                            datetime64[ns]
tipo_de_violência_denunciada            object
bairro_da_vítima                        object
faixa_etária                            object
raça/cor                                object
sexo                                    object
vínculo_do_suspeito                     object
ano                                      int64
dtype: object

--- Valores Nulos ---
protocolo                       0
data                            0
tipo_de_violência_denunciada    0
bairro_da_vítima                0
faixa_etária                    0
raça/cor                        0
sexo                            0
vínculo_do_suspeito             0
ano                             0
dtype: int64

--- Resumo Estatístico ---


,data,ano
count,1007,1007.00
mean,2024-10-04 15:42:21.807348480,2024.27
min,2023-03-14 00:00:00,2023.00
25%,2024-03-31 12:00:00,2024.00
50%,2024-11-25 00:00:00,2024.00
75%,2025-04-19 00:00:00,2025.00
max,2025-12-30 00:00:00,2025.00
std,NaN,0.75


In [27]:
df_agrupado = df_completo.groupby('tipo_de_violência_denunciada')[['bairro_da_vítima', 'ano']].value_counts().sort_values(ascending=False).reset_index(name='Total')

In [28]:
df_agrupado

,tipo_de_violência_denunciada,bairro_da_vítima,ano,Total
0,Abandono,Vila Morangueira,2025,9
1,Negligência,Zona 04,2024,9
2,Psíquica,Vila Morangueira,2025,9
3,Negligência,Jardim Alvorada,2024,9
4,Negligência,Vila Morangueira,2025,9
...,...,...,...,...
668,Maus Tratos,Zona 06,2023,1
669,Maus Tratos,Zona 02,2025,1
670,Maus Tratos,Zona 02,2023,1
671,Maus Tratos,Zona 01,2024,1
